# 🏆 Real-time Sports Highlight Generator — MULTIMODAL_003
**Team: team-3195 · indeevar.ravinuthala · AMD GPU Hackathon (ROCm)**

Multi-modal pipeline: **video frames (YOLOv8)** + **commentary (Whisper)** + **crowd noise (RMS energy)**
→ fused event detection → **highlight clips + captions + thumbnails**.

> ⚠️ GPU budget = 4 hrs / 24 hrs. Heavy cells are marked 🔥 (GPU). Setup cells are CPU-only.

## 0️⃣ Environment check (AMD ROCm)

In [ ]:
# Verify we are on the AMD GPU with ROCm
!rocm-smi || echo "rocm-smi not found (CPU fallback mode)"

import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))   # On ROCm this prints the AMD GPU
    print("HIP version:", getattr(torch.version, "hip", "n/a"))

## 1️⃣ Install dependencies (CPU-time, run once)

In [ ]:
# Torch is pre-installed in the ROCm env — do NOT reinstall it.
%pip install -q opencv-python-headless openai-whisper ultralytics "transformers>=4.40" accelerate Pillow moviepy soundfile matplotlib
!ffmpeg -version | head -1

## 2️⃣ Get a sample sports video (public dataset)
Per the rules: **public datasets only**. Options:
- Upload your own clip to `data/sample_video.mp4`
- SoccerNet (https://www.soccer-net.org/) — register for the free password
- Any CC-licensed sports clip

💡 Do this **before** your GPU session starts.

In [ ]:
import os
for d in ["data", "results", "output/highlights", "output/captions", "output/thumbnails"]:
    os.makedirs(d, exist_ok=True)

VIDEO = "data/sample_video.mp4"
assert os.path.exists(VIDEO), "⚠️ Place your sports video at data/sample_video.mp4 first!"
print("Video found:", VIDEO, f"({os.path.getsize(VIDEO)/1e6:.1f} MB)")

## 3️⃣ Extract frames + audio (CPU)

In [ ]:
import os, sys
# run from project root regardless of where the notebook lives
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import frame_extractor

frames = frame_extractor.extract_frames(VIDEO, "data/frames", fps=1.0)
audio_path = frame_extractor.extract_audio(VIDEO, "data/audio.wav")
print(f"{len(frames)} frames, audio at {audio_path}")

## 4️⃣ 🔥 Visual analysis — YOLOv8 on AMD GPU

In [ ]:
import visual_analyzer
visual = visual_analyzer.analyze_frames("data/frames", "results/visual.json", model_name="yolov8n.pt")
print("Top-5 most action-dense frames:")
for f in sorted(visual, key=lambda x: -x["visual_score"])[:5]:
    print(f"  t={f['timestamp']:>6.1f}s  score={f['visual_score']:.2f}  persons={f['persons']}")

## 5️⃣ 🔥 Audio analysis — Whisper ASR + crowd-energy spikes

In [ ]:
import audio_processor
audio = audio_processor.run("data/audio.wav", "results/audio.json", model_size="base")
print("\nSample excited commentary:")
for s in sorted(audio["segments"], key=lambda x: -x["excitement"])[:5]:
    print(f"  [{s['start']:>6.1f}s] ({s['excitement']:.2f}) {s['text']}")

## 6️⃣ Multi-modal event fusion
`S(t) = 0.45·visual + 0.35·commentary + 0.20·crowd` → adaptive-threshold peak picking

In [ ]:
import event_detector
payload = event_detector.run("results/visual.json", "results/audio.json", "results/events.json")

In [ ]:
# Visualize the fused excitement timeline
import matplotlib.pyplot as plt

scores = payload["fused_scores"]
plt.figure(figsize=(14, 4))
plt.plot(scores, lw=1.2, label="Fused multi-modal score")
plt.axhline(payload["threshold"], color="r", ls="--", label="Detection threshold")
for e in payload["events"]:
    plt.axvspan(e["start"], e["end"], alpha=0.25, color="orange")
    plt.annotate(f"#{e['event_id']} {e['event_type']}", (e["peak_time"], scores[e["peak_time"]]),
                 textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)
plt.xlabel("time (s)"); plt.ylabel("excitement"); plt.legend(); plt.title("Multi-modal Event Detection")
plt.tight_layout(); plt.savefig("results/timeline.png", dpi=120); plt.show()

## 7️⃣ Cut highlight clips (CPU/ffmpeg)

In [ ]:
import highlight_generator
events = highlight_generator.run(VIDEO, "results/events.json")
reel = highlight_generator.make_reel(events)

## 8️⃣ 🔥 Generate captions — BLIP + Whisper commentary fusion

In [ ]:
import caption_generator
# use_blip2=True for BLIP-2 OPT-2.7B if you have GPU budget; False = lightweight BLIP-base
events = caption_generator.run("results/events.json", "results/audio.json", "data/frames", use_blip2=False)

## 9️⃣ Create thumbnails (sharpest, most action-dense frame + banner)

In [ ]:
import thumbnail_creator
events = thumbnail_creator.run("results/events.json")

from IPython.display import Image as IPyImage, display
for e in events[:4]:
    print(f"\n=== Event #{e['event_id']} — {e['event_type']} ({e['timestamp_start']}–{e['timestamp_end']}) ===")
    print("Caption:", e["caption"])
    display(IPyImage(e["thumbnail"], width=600))

## 🔟 Final output package

In [ ]:
import json
with open("results/events.json") as f:
    payload = json.load(f)

final = [{k: e.get(k) for k in
          ["event_id","timestamp_start","timestamp_end","event_type",
           "confidence","caption","thumbnail","clip"]}
         for e in payload["events"]]

with open("output/final_highlights.json", "w") as f:
    json.dump(final, f, indent=2)
print(json.dumps(final, indent=2))
print("\n✅ Pipeline complete! Check output/ for highlights, captions, thumbnails.")

## 📝 Learnings & Future Work (fill in after your runs!)

**ROCm experience:**
- PyTorch on ROCm exposes the same `torch.cuda` API — YOLOv8, Whisper and BLIP ran without code changes.
- (Add your own notes: model load times, VRAM usage from `rocm-smi`, any HIP quirks…)

**Future work:**
- True real-time streaming (sliding-window inference on live feeds)
- LLM auto-commentary on detected events (vLLM serving on ROCm)
- Score-overlay OCR (pytesseract) as a 4th modality
- Vertical 9:16 social-media clips with burned-in subtitles
- Multi-sport fine-tuned action classifier instead of COCO heuristics